# Stage 3A - Direct LLM Benchmark (Improved)
## Thinking model (`deepseek-ai/DeepSeek-R1-Distill-Qwen-14B`) | 2-class benchmark

Improvements over the baseline run:

| Issue in baseline | Fix applied here |
|---|---|
| `max_tokens=128` truncates reasoning chain → Strict_Format_Rate only 48% | Raised to **512** |
| Prompt lacks concrete examples | Added **3-shot examples** with full reasoning format |
| No explicit reasoning guidance | Prompt asks model to reason briefly then output exactly `Final label: X` |

This notebook ran on Colab with GPU A100.

Saved outputs (same format as baseline):
- per-sample predictions CSV
- summary JSON
- one-row metrics CSV
- confusion matrix PNG

In [ ]:
!pip install -q vllm pandas scikit-learn matplotlib seaborn tqdm

from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi

In [ ]:
import os
import re
import json
import time
import numpy as np
import pandas as pd
from tqdm import tqdm

from vllm import LLM, SamplingParams
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
    mean_squared_error,
    mean_absolute_error,
)
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# ============================================
# Paths and basic configuration
# ============================================
DATA_PATH  = "/content/drive/MyDrive/Yelp_Project/data/processed/test_data.csv"
OUTPUT_DIR = "/content/drive/MyDrive/Yelp_Project/LLM/deepseek_r1_distill_qwen14b_2class_improved"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TEXT_COL  = "text"
LABEL_COL = "stars"

TASK_TYPE  = "2_class"
RUN_ID     = "deepseek_r1_distill_qwen14b_2class_improved"
MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"

BATCH_SIZE    = 16
MAX_MODEL_LEN = 4096
MAX_NEW_TOKENS = 512   # ← was 128; reasoning model needs room to finish CoT

LABELS         = [0, 1]
DISPLAY_LABELS = ["Negative (0)", "Positive (1)"]

In [ ]:
# ============================================
# Load and prepare the test set
# ============================================
df_raw = pd.read_csv(DATA_PATH)
df_raw = df_raw[[TEXT_COL, LABEL_COL]].copy()
df_raw[LABEL_COL] = df_raw[LABEL_COL].astype(int)

# Drop 3-star reviews (same as baseline)
df_test = df_raw[df_raw[LABEL_COL] != 3].copy()
df_test["original_stars"] = df_test[LABEL_COL]
df_test["label"] = df_test[LABEL_COL].apply(lambda x: 0 if x <= 2 else 1)
EVAL_LABEL_COL = "label"

print(f"Loaded {len(df_test)} samples for {TASK_TYPE}")
print(df_test[[TEXT_COL, "original_stars", "label"]].head())

In [ ]:
# ============================================
# Improved prompt with 3-shot examples
# ============================================

# Examples are representative but short to avoid prompt bloat.
# One negative, one positive, one tricky case (mixed but ultimately positive).
FEW_SHOT_EXAMPLES = """Here are three examples:

Review: "The food was cold, the service was rude, and I waited 45 minutes for a simple order. Absolutely terrible experience."
Reasoning: Strong negative language throughout — cold food, rude service, long wait. This is clearly a negative review.
Final label: 0

Review: "Amazing brunch! The eggs benedict were perfectly poached and the mimosas were generous. Will definitely be back."
Reasoning: Enthusiastic praise, specific positive details, intent to return. Clearly positive.
Final label: 1

Review: "The decor is a bit dated but the food more than makes up for it — best tacos I've had in years."
Reasoning: Minor complaint about decor, but the dominant sentiment is strong praise for the food. Overall positive.
Final label: 1
"""

def build_prompt(text: str) -> str:
    return (
        "You are a Yelp review sentiment classifier.\n"
        "Task: classify the review as 0 (Negative, original 1-2 stars) or 1 (Positive, original 4-5 stars).\n\n"
        + FEW_SHOT_EXAMPLES +
        "Now classify the following review.\n"
        "Reason briefly (2-3 sentences), then output EXACTLY this as your LAST line:\n"
        "Final label: X\n"
        "where X is 0 or 1. Do not add anything after that line.\n\n"
        f"Review: {text}"
    )

def parse_output(text: str):
    text = str(text).strip()
    # Strict: last occurrence of 'Final label: X' wins
    matches = re.findall(r"Final label:\s*([01])", text, flags=re.IGNORECASE)
    if matches:
        return int(matches[-1]), True
    # Fallback: last standalone 0 or 1
    binary_matches = re.findall(r"\b([01])\b", text)
    if binary_matches:
        return int(binary_matches[-1]), False
    lowered = text.lower()
    if "negative" in lowered:
        return 0, False
    if "positive" in lowered:
        return 1, False
    return None, False

# Sanity check
sample = build_prompt("This place was fantastic!")
print(sample[:600])

In [ ]:
# ============================================
# Model setup
# ============================================
sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=MAX_NEW_TOKENS,   # 512 instead of 128
)

llm = LLM(
    model=MODEL_NAME,
    trust_remote_code=True,
    max_model_len=MAX_MODEL_LEN,
)

In [ ]:
# ============================================
# Batched inference
# ============================================
all_rows = []
start_time = time.time()

for start in tqdm(range(0, len(df_test), BATCH_SIZE), desc="Inference"):
    end = min(start + BATCH_SIZE, len(df_test))
    batch_df = df_test.iloc[start:end].copy()

    prompts = [build_prompt(text) for text in batch_df[TEXT_COL].tolist()]
    outputs = llm.generate(prompts, sampling_params)

    for i, output in enumerate(outputs):
        raw_text = output.outputs[0].text
        pred, strict_ok = parse_output(raw_text)

        all_rows.append({
            "text"            : batch_df.iloc[i][TEXT_COL],
            "original_stars"  : int(batch_df.iloc[i]["original_stars"]),
            "true_label"      : int(batch_df.iloc[i][EVAL_LABEL_COL]),
            "raw_output"      : raw_text,
            "pred"            : pred,
            "strict_format_ok": strict_ok,
        })

eval_time_seconds = time.time() - start_time

result_df = pd.DataFrame(all_rows)
pred_path = os.path.join(OUTPUT_DIR, f"{RUN_ID}_predictions.csv")
result_df.to_csv(pred_path, index=False, encoding="utf-8")

print(f"Saved predictions to: {pred_path}")
print(f"Eval time (s): {eval_time_seconds:.2f}")
print(result_df.head())

In [ ]:
# ============================================
# Evaluation
# ============================================
eval_df = result_df.dropna(subset=["pred"]).copy()
eval_df["pred"] = eval_df["pred"].astype(int)

y_true = eval_df["true_label"].to_numpy()
y_pred = eval_df["pred"].to_numpy()

accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")
cm       = confusion_matrix(y_true, y_pred, labels=LABELS)
mse      = mean_squared_error(y_true, y_pred)
mae      = mean_absolute_error(y_true, y_pred)

valid_prediction_rate = result_df["pred"].notna().mean()
strict_format_rate    = result_df["strict_format_ok"].mean()
eval_speed            = len(result_df) / eval_time_seconds if eval_time_seconds > 0 else np.nan

report = classification_report(
    y_true, y_pred, labels=LABELS, output_dict=True, digits=4
)

summary = {
    "Run_ID"               : RUN_ID,
    "Task_Type"            : TASK_TYPE,
    "Mode"                 : "Prompt_Inference_Improved",
    "Model"                : MODEL_NAME,
    "Batch_Size"           : BATCH_SIZE,
    "Max_New_Tokens"       : MAX_NEW_TOKENS,
    "Few_Shot"             : True,
    "Accuracy"             : float(accuracy),
    "Macro_F1"             : float(macro_f1),
    "Valid_Prediction_Rate": float(valid_prediction_rate),
    "Strict_Format_Rate"   : float(strict_format_rate),
    "Off_By_1_Acc"         : None,
    "Adj_Error_Ratio"      : None,
    "Mid_Confusion_Ratio"  : None,
    "MSE"                  : float(mse),
    "MAE"                  : float(mae),
    "Eval_Time(s)"         : float(eval_time_seconds),
    "Eval_Speed(s/s)"      : float(eval_speed),
    "Confusion_Matrix"     : cm.tolist(),
    "Classification_Report": report,
}

summary_path = os.path.join(OUTPUT_DIR, f"{RUN_ID}_summary.json")
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

metrics_row = pd.DataFrame([{
    "Run_ID"               : RUN_ID,
    "Task_Type"            : TASK_TYPE,
    "Mode"                 : "Prompt_Inference_Improved",
    "Model"                : MODEL_NAME,
    "Batch_Size"           : BATCH_SIZE,
    "Max_New_Tokens"       : MAX_NEW_TOKENS,
    "Few_Shot"             : True,
    "Accuracy"             : accuracy,
    "Macro_F1"             : macro_f1,
    "Valid_Prediction_Rate": valid_prediction_rate,
    "Strict_Format_Rate"   : strict_format_rate,
    "Off_By_1_Acc"         : np.nan,
    "Adj_Error_Ratio"      : np.nan,
    "Mid_Confusion_Ratio"  : np.nan,
    "MSE"                  : mse,
    "MAE"                  : mae,
    "Eval_Time(s)"         : eval_time_seconds,
    "Eval_Speed(s/s)"      : eval_speed,
}])

metrics_csv_path = os.path.join(OUTPUT_DIR, f"{RUN_ID}_metrics.csv")
metrics_row.to_csv(metrics_csv_path, index=False, encoding="utf-8")

print("Saved summary JSON to:", summary_path)
print("Saved metrics CSV to :", metrics_csv_path)
metrics_row

In [ ]:
# ============================================
# Confusion matrix
# ============================================
plt.figure(figsize=(7, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=DISPLAY_LABELS,
    yticklabels=DISPLAY_LABELS,
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title(f"Confusion Matrix - {RUN_ID}")
plt.tight_layout()

fig_path = os.path.join(OUTPUT_DIR, f"{RUN_ID}_confusion_matrix.png")
plt.savefig(fig_path, dpi=200)
plt.show()
print("Saved confusion matrix to:", fig_path)

In [ ]:
# ============================================
# Comparison with baseline run
# ============================================
baseline = {
    "Run_ID"               : "deepseek_r1_distill_qwen14b_2class_fulltest (baseline)",
    "Max_New_Tokens"       : 128,
    "Few_Shot"             : False,
    "Accuracy"             : 0.8004,
    "Macro_F1"             : 0.7943,
    "Valid_Prediction_Rate": 0.9369,
    "Strict_Format_Rate"   : 0.4816,
}
improved = {
    "Run_ID"               : f"{RUN_ID} (improved)",
    "Max_New_Tokens"       : MAX_NEW_TOKENS,
    "Few_Shot"             : True,
    "Accuracy"             : accuracy,
    "Macro_F1"             : macro_f1,
    "Valid_Prediction_Rate": valid_prediction_rate,
    "Strict_Format_Rate"   : strict_format_rate,
}

cmp_df = pd.DataFrame([baseline, improved]).set_index("Run_ID")
print("\n=== Baseline vs Improved ===")
print(cmp_df.to_string())

In [ ]:
# ============================================
# Final console summary
# ============================================
print(f"Run_ID               : {RUN_ID}")
print(f"Task_Type            : {TASK_TYPE}")
print(f"Mode                 : Prompt_Inference_Improved")
print(f"Model                : {MODEL_NAME}")
print(f"Max_New_Tokens       : {MAX_NEW_TOKENS}")
print(f"Few_Shot             : True")
print(f"Accuracy             : {accuracy:.4f}")
print(f"Macro_F1             : {macro_f1:.4f}")
print(f"Valid_Prediction_Rate: {valid_prediction_rate:.4f}")
print(f"Strict_Format_Rate   : {strict_format_rate:.4f}")
print(f"MSE                  : {mse:.6f}")
print(f"MAE                  : {mae:.6f}")
print(f"Eval_Time(s)         : {eval_time_seconds:.2f}")
print(f"Eval_Speed(s/s)      : {eval_speed:.4f}")